# GSEP Mapping Analysis

This notebook analyzes Gas System Enhancement Program (GSEP) investments in Massachusetts,
examining their overlap with equity-focused regions:

- **Environmental Justice Communities (EJC)**
- **Low-to-Moderate Income (LMI) areas**
- **Gateway Cities**

## Setup

In [ ]:
# Install dependencies if needed (uncomment if running in Colab)
# !pip install contextily geopandas

In [ ]:
import matplotlib.pyplot as plt

from src import (
    load_gsep_projects,
    load_ejc_data,
    load_lmi_data,
    load_gateway_cities,
    load_national_grid_feeders,
    compute_point_overlap,
    compute_line_length_in_region,
    add_equity_region_flags,
    compute_equity_overlap_counts,
    plot_overlay_map,
    plot_overlap_bar_chart,
    plot_equity_heatmap,
    plot_equity_overlap_histogram,
)

## Load Data

Data is automatically cached after the first download.

In [ ]:
# Load GSEP projects (downloads from ArcGIS if not cached)
gsep_df = load_gsep_projects()
print(f"Loaded {len(gsep_df)} GSEP projects")

In [ ]:
# Load equity region layers
ejc_df = load_ejc_data()
lmi_df = load_lmi_data()
gateway_df = load_gateway_cities()
national_grid_df = load_national_grid_feeders()

## GSEP Analysis: Environmental Justice Communities

In [ ]:
# Map GSEP projects with EJC overlay
plot_overlay_map(
    ejc_df, gsep_df,
    title="Planned 2026-2029 GSEP Projects by LDC",
    region_color="#00ff0044",
    region_label="EJCs",
    overlay_column="LDC",
)
plt.show()

In [ ]:
# Compute and plot EJC overlap
gsep_df, ejc_stats = compute_point_overlap(gsep_df, ejc_df, "EJC")
plot_overlap_bar_chart(
    ejc_stats,
    title="Location of Planned GSEP Projects (2026-2029)",
    ylabel="Number of GSEP projects",
)
plt.show()

## GSEP Analysis: Low-to-Moderate Income Areas

In [ ]:
# Filter LMI data to only LMI block groups
lmi_only = lmi_df[lmi_df["is_lmi"]]

# Map GSEP with LMI overlay
plot_overlay_map(
    lmi_only, gsep_df,
    title="Planned 2026-2029 GSEP Projects by LDC",
    region_color="#9448BC44",
    region_label="LMI Census Block Groups",
    overlay_column="LDC",
)
plt.show()

In [ ]:
# Compute and plot LMI overlap
gsep_df, lmi_stats = compute_point_overlap(gsep_df, lmi_only, "LMI")
plot_overlap_bar_chart(
    lmi_stats,
    title="Location of Planned GSEP Projects (2026-2029)",
    ylabel="Number of GSEP projects",
)
plt.show()

## GSEP Analysis: Gateway Cities

In [ ]:
# Map GSEP with Gateway Cities overlay
plot_overlay_map(
    gateway_df, gsep_df,
    title="Planned 2026-2029 GSEP Projects by LDC",
    region_color="#a7260844",
    region_label="Gateway Cities",
    overlay_column="LDC",
)
plt.show()

In [ ]:
# Compute and plot Gateway Cities overlap
gsep_df, gateway_stats = compute_point_overlap(gsep_df, gateway_df, "Gateway")
plot_overlap_bar_chart(
    gateway_stats,
    title="Location of Planned GSEP Projects (2026-2029)",
    ylabel="Number of GSEP projects",
)
plt.show()

## National Grid NWA Opportunities Analysis

In [ ]:
# Map NWA opportunities with EJC overlay
plot_overlay_map(
    ejc_df, national_grid_df,
    title="Announced 2026-2029 NWA Opportunities (National Grid)",
    region_color="#00ff0044",
    region_label="EJCs",
    overlay_filter_col="nwa_opportunity",
    show_legend=False,
)
plt.show()

In [ ]:
# Compute NWA feeder length in EJCs
national_grid_df, ejc_length_stats = compute_line_length_in_region(
    national_grid_df, ejc_df, "EJC",
    filter_col="nwa_opportunity", filter_value=True
)
plot_overlap_bar_chart(
    ejc_length_stats,
    title="National Grid Announced NWA Opportunities (2026-2029)",
    ylabel="Miles of feeder",
)
plt.show()

In [ ]:
# Compute NWA feeder length in LMI areas
national_grid_df, lmi_length_stats = compute_line_length_in_region(
    national_grid_df, lmi_only, "LMI",
    filter_col="nwa_opportunity", filter_value=True
)
plot_overlap_bar_chart(
    lmi_length_stats,
    title="National Grid Announced NWA Opportunities (2026-2029)",
    ylabel="Miles of feeder",
)
plt.show()

In [ ]:
# Compute NWA feeder length in Gateway Cities
national_grid_df, gateway_length_stats = compute_line_length_in_region(
    national_grid_df, gateway_df, "Gateway_Cities",
    filter_col="nwa_opportunity", filter_value=True
)
plot_overlap_bar_chart(
    gateway_length_stats,
    title="National Grid Announced NWA Opportunities (2026-2029)",
    ylabel="Miles of feeder",
)
plt.show()

## Combined Equity Analysis

Analyze GSEP projects by how many equity regions they overlap with.

In [ ]:
# Add all equity region flags
equity_layers = {
    "EJC": ejc_df,
    "LMI": lmi_only,
    "Gateway": gateway_df,
}
gsep_df = add_equity_region_flags(gsep_df, equity_layers)

# Count overlaps
flag_cols = ["is_in_ejc", "is_in_lmi", "is_in_gateway"]
gsep_df["equity_region_overlaps"] = compute_equity_overlap_counts(gsep_df, flag_cols)

In [ ]:
# Heatmap of equity region overlaps
plot_equity_heatmap(
    gsep_df,
    "equity_region_overlaps",
    title="Planned 2026-2029 GSEP Projects by Overlap with Equity Regions",
)
plt.show()

In [ ]:
# Histogram of equity region overlaps
plot_equity_overlap_histogram(
    gsep_df,
    "equity_region_overlaps",
    title="Planned GSEP Projects (2026-2029)",
)
plt.show()

## Adding New Equity Layers

The system is designed to be easily extensible. To add a new equity layer:

1. Place the GeoJSON/shapefile in `data/raw/`
2. Add a loader function in `src/data_loader.py`
3. Use the existing analysis functions:

```python
# Example: Adding a new "Opportunity Zone" layer
from src import load_opportunity_zones, compute_point_overlap

oz_df = load_opportunity_zones()
gsep_df, oz_stats = compute_point_overlap(gsep_df, oz_df, "Opportunity Zone")
plot_overlap_bar_chart(oz_stats, title="GSEP in Opportunity Zones")
```